In [8]:
import requests
from datetime import datetime
import os
from dotenv import load_dotenv
import pandas as pd
import json
load_dotenv()
api_key = os.getenv("API_KEY")

In [ ]:
class ExtractChampionData:

    def __init__(self):
        self.last_version = self.get_latest_version()
        self.list_champ = self.get_all_champ_general_data()
        self.list_champ = list(self.list_champ.keys())

    def get_latest_version(self):
        """Get the last version of the game"""
        url = "https://ddragon.leagueoflegends.com/api/versions.json"
        rep = requests.get(url).json()
        latest = rep[0]
        return latest

    def get_all_champ_general_data(self):
        """Get the champion data"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion.json"
        resp = requests.get(url).json()
        return resp['data']
    
    def get_details_champ_data(self):
        """Get detail champion data"""
        data = []
        for champ in self.list_champ:
            url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion/{champ}.json"
            resp = requests.get(url).json()['data']
            data.append(resp[champ])
        return data
    
    def get_champion_mastery(self, puuid : str):
        """Get all champion mastery entries sorted by number of champion points descending."""

        api_url = f"https://euw1.api.riotgames.com/lol/champion-mastery/v4/champion-masteries/by-puuid/{puuid}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
        

    # def download_all_png_champion(self):
    #     """Download all champions pictures"""
    #     for champ in self.list_champ:
    #         url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/img/champion/{champ}.png"
    #         path_save = fr"C:\Users\najim\Documents\Projets\LeagueOfLegends\images\{champ}.png"

    #         resp = requests.get(url)
    #         with open(path_save, "wb") as file:
    #             file.write(resp.content)

In [10]:
class TransformChampionData:

    def __init__(self):
        self.keys_to_drop = ['image', 'skins', 'blurb']
        self.table_champion = ['key', 'name', 'title', 'lore', 'tags', 'partype']
        self.table_champ_passive = ['key', 'passive']
        self.table_champ_info = ['key', 'info', 'allytips', 'enemytips']
        self.table_champ_spells = ['key', 'spells']
        self.table_champ_stats = ['key', 'stats']
    
    def drop_keys(self, data : list) -> list:
        new_data = []
        for dict_champ in data:
            dict_champ = {k: v for k,v in dict_champ.items() if k not in self.keys_to_drop}
            new_data.append(dict_champ)
        return new_data
    
    def dispatch_data(self, data : list):
        table_champion = []
        table_champ_passive = []
        table_champ_info = []
        table_champ_spells = []
        table_champ_stats = []
        
        for dict_champ in data:
            champion = {k: v for k,v in dict_champ.items() if k in self.table_champion}
            champ_passive = {k: v for k,v in dict_champ.items() if k in self.table_champ_passive}
            champ_info = {k: v for k,v in dict_champ.items() if k in self.table_champ_info}
            champ_spells = {k: v for k,v in dict_champ.items() if k in self.table_champ_spells}
            champ_stats = {k: v for k,v in dict_champ.items() if k in self.table_champ_stats}

            table_champion.append(champion)
            table_champ_passive.append(champ_passive)
            table_champ_info.append(champ_info)
            table_champ_spells.append(champ_spells)
            table_champ_stats.append(champ_stats)
            
        return table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats

In [ ]:
def pipeline_champion():
    details = ExtractChampionData().get_details_champ_data()
    details = TransformChampionData().drop_keys(details)
    return details

champions_data = pipeline_champion()
champions_data

In [11]:
table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats = TransformChampionData().dispatch_data(champions_data)

In [13]:
pd.DataFrame(table_champion)

,key,name,title,lore,tags,partype
0,266,Aatrox,Épée des Darkin,"Autrefois, Aatrox et ses frères étaient honoré...",[Fighter],Puits de sang
1,103,Ahri,Renarde à neuf queues,"Connectée à la magie du royaume spirituel, Ahr...","[Mage, Assassin]",Mana
2,84,Akali,Assassin rebelle,Ayant abandonné l'Ordre Kinkou et le titre de ...,[Assassin],Énergie
3,166,Akshan,Sentinelle rebelle,"Se jouant du danger, Akshan combat le mal sans...","[Marksman, Assassin]",Mana
4,12,Alistar,Minotaure,Alistar est un guerrier redoutable cherchant à...,"[Tank, Support]",Mana
...,...,...,...,...,...,...
167,221,Zeri,Étincelle de Zaun,Zeri est une jeune femme vive et téméraire ori...,[Marksman],Mana
168,115,Ziggs,Expert des Hexplosifs,Amoureux des grosses bombes et des mèches cour...,[Mage],Mana
169,26,Zilean,Gardien du Temps,"Naguère puissant mage d'Icathia, Zilean devint...","[Support, Mage]",Mana
170,142,Zoé,Manifestation du Crépuscule,"Incarnation de l'espièglerie, de l'imagination...",[Mage],Mana
